In [2]:
"""
GOAL SAVER - PayPal AI Hackathon project
-----------------------------------------
Install :  pip install streamlit pandas numpy requests
Run     :  streamlit run app.py

Optional keys (set as environment variables):
  PAYPAL_CLIENT_ID, PAYPAL_SECRET   -> from developer.paypal.com (Sandbox app)
  ANTHROPIC_API_KEY                 -> for AI advice (any AI tool is allowed)
"""
import os
import numpy as np
import pandas as pd
import requests
import streamlit as st

CUR = "$"  # PayPal sandbox works best with USD
WANTS = ["Food Delivery", "Shopping", "Entertainment", "Travel"]  # categories we may suggest cutting

st.set_page_config(page_title="Goal Saver", page_icon="🎯")
st.title("🎯 Goal Saver: AI savings tracker")


2026-11-11 17:32:54.791 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 17:32:54.791 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 17:32:55.240 
  command:

    streamlit run c:\Users\DELL\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-11-11 17:32:55.251 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 17:32:55.251 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [3]:


# ---------- STEP 1: GET SPENDING DATA ----------
def demo_data():
    """Fake bank data so the app works without any upload."""
    rng = np.random.default_rng(1)
    base = {"Rent": 400, "Food": 150, "Food Delivery": 90,
            "Shopping": 110, "Entertainment": 60, "Travel": 50}
    rows = []
    for month in pd.period_range("2026-01", "2026-10", freq="M"):
        for cat, amt in base.items():
            rows.append({"month": str(month), "category": cat,
                         "amount": round(amt * rng.uniform(0.8, 1.3), 2)})
    return pd.DataFrame(rows)


st.sidebar.header("Your money")
income = st.sidebar.number_input("Monthly income", value=1000)
file = st.sidebar.file_uploader("Bank statement CSV (date, category, amount)", type="csv")

if file:
    df = pd.read_csv(file)
    df["month"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m")
else:
    df = demo_data()
    st.sidebar.info("Using demo data. Upload your own CSV to replace it.")

spend_per_month = df.groupby("month")["amount"].sum()
saved_per_month = income - spend_per_month  # what was left each month

# ---------- STEP 2: SET THE GOAL ----------
st.header("1. Your goal")
c1, c2 = st.columns(2)
goal_name = c1.text_input("What are you saving for?", "Smartphone")
goal_amount = c2.number_input("Goal amount", value=3000)
months = c1.number_input("In how many months?", min_value=1, value=6)
months_done = c2.number_input("Months already passed", min_value=0, value=2)
saved_so_far = st.number_input("Saved so far", value=900)

required = goal_amount / months  # money to save every month
chance = (saved_per_month >= required).mean() * 100  # % of past months we beat the target

# ---------- STEP 3: PROGRESS + PROBABILITY ----------
st.header("2. How am I doing?")
st.write(f"You must save **{CUR}{required:,.0f} every month**.")
st.progress(min(saved_so_far / goal_amount, 1.0))
st.write(f"Saved {CUR}{saved_so_far:,.0f} of {CUR}{goal_amount:,.0f}")
st.metric("Chance of reaching the goal", f"{chance:.0f}%",
          help="Share of your past months where you saved at least the monthly target")

expected = required * months_done
behind = expected - saved_so_far
if behind <= 0:
    st.success("✅ You are on track. Keep going!")
else:
    st.warning(f"⚠️ You are {CUR}{behind:,.0f} behind schedule.")

    # ---------- STEP 4: SUGGEST WHAT TO CUT (ONLY IF USER APPROVES) ----------
    months_left = max(months - months_done, 1)
    extra_needed = behind / months_left  # extra saving needed each month
    avg_by_cat = df[df["category"].isin(WANTS)].groupby("category")["amount"].mean()
    top_cat = avg_by_cat.idxmax()  # the "want" where we spend the most
    cut = min(extra_needed, avg_by_cat[top_cat])
    st.info(f"💡 Suggestion: spend {CUR}{cut:,.0f} less per month on **{top_cat}** "
            f"to catch up in {months_left} months.")
    if st.button("✅ Approve this plan"):
        st.success(f"Plan approved! New {top_cat} budget: "
                   f"{CUR}{avg_by_cat[top_cat] - cut:,.0f}/month")

# ---------- STEP 5: PRICE TRACKER ----------
st.header("3. Price watcher")


def get_price_history(item):
    """Fake last 6 months of prices.
    TODO: replace this with a real Channel3 (or any shopping) API call."""
    rng = np.random.default_rng(sum(map(ord, item)))
    start = 300 + rng.integers(0, 500)
    return [start + i * 8 + int(rng.integers(-12, 12)) for i in range(6)]


prices = get_price_history(goal_name)
slope, intercept = np.polyfit(range(6), prices, 1)  # simple straight-line trend
next_price = slope * 6 + intercept
st.line_chart(pd.DataFrame({"price": prices}))
if slope > 0:
    st.error(f"📈 Price is rising. Estimated next month: {CUR}{next_price:,.0f} (just an estimate)")
else:
    st.success(f"📉 Price is falling. Estimated next month: {CUR}{next_price:,.0f} (just an estimate)")


# ---------- STEP 6: PAYPAL SANDBOX PAYMENT ----------
def paypal_checkout_link(amount):
    """Create a PayPal sandbox order and return the link where the buyer approves it."""
    cid, secret = os.getenv("PAYPAL_CLIENT_ID"), os.getenv("PAYPAL_SECRET")
    base = "https://api-m.sandbox.paypal.com"
    tok = requests.post(f"{base}/v1/oauth2/token", auth=(cid, secret),
                        data={"grant_type": "client_credentials"}).json()["access_token"]
    body = {"intent": "CAPTURE",
            "purchase_units": [{"amount": {"currency_code": "USD", "value": f"{amount:.2f}"}}],
            "payment_source": {"paypal": {"experience_context": {
                "return_url": "https://example.com/success",
                "cancel_url": "https://example.com/cancel"}}}}
    r = requests.post(f"{base}/v2/checkout/orders", json=body,
                      headers={"Authorization": f"Bearer {tok}"})
    r.raise_for_status()
    return next(l["href"] for l in r.json()["links"] if l["rel"] in ("payer-action", "approve"))


st.header("4. Buy it with PayPal")
if saved_so_far >= prices[-1]:
    st.success("🎉 You have enough saved to buy it now!")
if st.button(f"Reserve {goal_name} with PayPal (sandbox)"):
    try:
        st.markdown(f"[👉 Click to pay in PayPal sandbox]({paypal_checkout_link(prices[-1])})")
    except Exception as e:
        st.error(f"Add your PayPal sandbox keys first. ({e})")



2026-11-11 20:40:37.820 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 20:40:37.822 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 20:40:37.823 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 20:40:37.826 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 20:40:37.828 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 20:40:37.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 20:40:37.833 Session state does not function when running a script without `streamlit run`
2026-11-11 20:40:37.835 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-11 20:40

ImportError: pyarrow requires NumPy 2.0 or newer, found 1.26.4

In [ ]:

# ---------- STEP 7: AI ADVICE ----------
def ask_ai(summary):
    """Ask an AI model for friendly advice. Falls back to a simple message with no key."""
    key = os.getenv("ANTHROPIC_API_KEY")
    if not key:
        return "Set ANTHROPIC_API_KEY to get AI advice. Tip: cut your biggest 'want' category first."
    r = requests.post("https://api.anthropic.com/v1/messages",
                      headers={"x-api-key": key, "anthropic-version": "2023-06-01",
                               "content-type": "application/json"},
                      json={"model": "claude-sonnet-4-6", "max_tokens": 300,
                            "messages": [{"role": "user", "content":
                                          "You are a friendly savings coach. In under 80 words, "
                                          "give practical advice. Not financial advice.\n" + summary}]})
    return r.json()["content"][0]["text"]


st.header("5. AI coach")
if st.button("Get AI advice"):
    info = (f"Goal: {goal_name} {goal_amount} in {months} months. Saved {saved_so_far}. "
            f"Must save {required:.0f}/month. Chance {chance:.0f}%. "
            f"Average monthly spend by category: {df.groupby('category')['amount'].mean().round(0).to_dict()}. "
            f"Price trend per month: {slope:.1f}.")
    st.write(ask_ai(info))

st.caption("Educational only. Not financial advice.")